# Whisper sencillo y gratis

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanrlg/whisper-toolkit/blob/simplify-long-audio/Whisper_Simple_Gratis.ipynb)

**Tu transcripción de siempre con menos pasos, sin contratar una API ni introducir datos de pago.**
Para audios de una hora o una hora y media. Ejecuta la única celda: instala lo necesario,
te pide el audio, transcribe y descarga un ZIP con **TXT, JSON, SRT y VTT**.

## Cómo usarlo

1. En Colab, ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**,
   si está disponible en tu cuenta gratuita. No necesitas Colab Pro. También puede usar CPU,
   pero con audios largos será más lento.
2. Deja `SEPARAR_HABLANTES` desactivado al empezar: **transcribir no necesita ninguna clave**.
3. Pulsa **▶**, autoriza Drive para guardar avances y selecciona tu audio.
   Para un archivo que ya está en Drive, puedes poner su ruta en `RUTA_AUDIO`, por ejemplo
   `/content/drive/MyDrive/Audios/reunion.m4a`.

`IDIOMA = es` está preparado para español. Usa `auto` si no sabes el idioma, o `en` para inglés.
Para grabaciones que alternan idiomas, revisa especialmente el resultado: se usa un idioma principal.

## Si quieres separar los hablantes — también sin API de pago

Esta función es opcional. Usa **pyannote Community-1**, ejecutado en el entorno de Colab.

1. Inicia sesión en Hugging Face y acepta las condiciones de
   [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1).
2. Crea un [token de lectura gratuito](https://huggingface.co/settings/tokens).
   Si usas permisos detallados, permite leer los modelos restringidos a los que tienes acceso.
   No necesitas una cuenta de pago ni una clave de pyannoteAI.
3. En el icono de llave de Colab (**Secretos**), guarda el token como `HF_TOKEN` y permite
   su acceso a este notebook. También reconoce secretos anteriores llamados `HUGGINGFACE_TOKEN`
   y `HUGGINGFACE_HUB_TOKEN`.
4. Activa `SEPARAR_HABLANTES` y vuelve a ejecutar con el mismo audio.
   **La transcripción ya guardada se reutiliza.**

No se escribe el token dentro del notebook ni de los resultados. Si falta o la diarización falla,
recibirás la transcripción sin hablantes con un mensaje explicando lo que falta.

## Qué se ha simplificado

- **Una sola transcripción** con faster-whisper `large-v3-turbo`. Sin segunda pasada de WhisperX,
  sin modelo de alineación adicional y sin las secciones de Groq.
- Instalación aislada del kernel, Python 3.12 y versiones fijadas: no hace falta reiniciar Colab
  tras instalar. La primera preparación y la descarga de modelos pueden tardar varios minutos.
- Se guarda cada bloque de aproximadamente cinco minutos. Si algo se interrumpe, vuelve a ejecutar
  con el mismo archivo y se recuperan los bloques terminados. El bloque interrumpido se repite.
- Cada bloque tiene dos segundos de contexto adicional; se conservan las palabras por su punto medio
  para reducir cortes y repeticiones en las uniones. Los timestamps siguen referidos al audio original.
- La diarización usa **la grabación completa** para mantener las etiquetas de hablante a lo largo
  de toda la reunión. Transcripción y diarización corren en procesos separados para liberar la memoria.
- Los hablantes se escriben en TXT, JSON, SRT y VTT. Si la diarización falla, no se pierde el texto.

## Dónde queda guardado

Con `GUARDAR_EN_DRIVE` activado: **Mi unidad → Transcripciones_Whisper**.
Cada audio/configuración tiene una carpeta de recuperación. Conserva esa carpeta y usa la misma
configuración para retomar. Si desactivas Drive, los avances desaparecen cuando Colab elimina la sesión.
Los resultados parciales se identifican como tales en TXT y JSON; verifica `complete` en JSON.

Los audios se procesan en Colab, sin enviarlos a una API de transcripción de pago.
Se requiere internet para instalar paquetes y descargar los modelos. Colab gratuito tiene
límites variables y no garantiza disponibilidad de GPU o duración de sesión.

Los tiempos de palabras y las voces detectadas pueden contener errores, sobre todo con ruido,
voces simultáneas y en las uniones de bloques. `SPEAKER_00`, etc. distingue voces, no nombres reales.

**Validación:** pruebas locales de recuperación y exportaciones, más una transcripción real corta
en CPU con un modelo pequeño. Falta probar GPU de Colab, diarización con un token autorizado y
un audio real de 60–90 minutos; no se garantiza un tiempo de ejecución.

Fuentes: [faster-whisper](https://github.com/SYSTRAN/faster-whisper),
[Community-1](https://huggingface.co/pyannote/speaker-diarization-community-1),
[límites de Colab](https://research.google.com/colaboratory/faq.html).


In [ ]:
# @title Transcribir con Whisper — sin API de pago
# @markdown Activa **Entorno de ejecución → Cambiar tipo → T4 GPU**, si está disponible.
# @markdown Deja la ruta vacía para subir el archivo desde tu computadora.
RUTA_AUDIO = "" # @param {type:"string"}
IDIOMA = "es" # @param ["es", "auto", "en"]
SEPARAR_HABLANTES = False # @param {type:"boolean"}
GUARDAR_EN_DRIVE = True # @param {type:"boolean"}


import hashlib
import os
from pathlib import Path
WORKER_SOURCE = '"""Whisper en Colab, sin API de pago. Motor autocontenido del notebook."""\nimport argparse\nfrom bisect import bisect_right\nimport hashlib\nimport html\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport zipfile\n\nSAMPLE_RATE = 16000\nDIARIZATION_MODEL = "pyannote/speaker-diarization-community-1"\nALGORITHM_VERSION = 1\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef save_json(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef prepare_job(audio_path, output_root, model="large-v3-turbo", language="es", chunk_seconds=300):\n    path = Path(audio_path)\n    if not path.is_file() or path.stat().st_size == 0:\n        raise ValueError("Selecciona un archivo de audio o video que no esté vacío.")\n    config = {"model": model, "language": language, "chunk_seconds": chunk_seconds,\n              "algorithm_version": ALGORITHM_VERSION, "faster_whisper_version": "1.2.1"}\n    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())\n    with path.open("rb") as source:\n        for block in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(block)\n    folder = Path(output_root) / digest.hexdigest()[:24]\n    folder.mkdir(parents=True, exist_ok=True)\n    save_json(folder / "config.json", config)\n    return folder\n\n\ndef chunk_plan(duration, size=300, context=2):\n    if duration <= 0 or size <= 0 or context < 0:\n        raise ValueError("Duración o tamaño de bloque no válido.")\n    return [dict(index=i, start=i * size, end=min(duration, (i + 1) * size),\n                 read_start=max(0, i * size - context),\n                 read_end=min(duration, (i + 1) * size + context))\n            for i in range(math.ceil(duration / size))]\n\n\ndef keep_core_segments(segments, chunk):\n    """Conserva cada palabra en el bloque al que pertenece su punto medio."""\n    kept = []\n    offset = chunk["read_start"]\n    for segment in segments:\n        words = []\n        for word in segment.get("words") or []:\n            start, end = word["start"] + offset, word["end"] + offset\n            midpoint = (start + end) / 2\n            if chunk["start"] <= midpoint < chunk["end"]:\n                words.append({**word, "start": max(0, start), "end": end})\n        if words:\n            kept.append({"start": words[0]["start"], "end": words[-1]["end"],\n                         "text": "".join(w["word"] for w in words).strip(), "words": words})\n        elif not segment.get("words"):\n            start, end = segment["start"] + offset, segment["end"] + offset\n            if chunk["start"] <= (start + end) / 2 < chunk["end"]:\n                kept.append({"start": start, "end": end, "text": segment["text"].strip(), "words": []})\n    return kept\n\n\ndef timestamp(seconds, separator="."):\n    total = max(0, round(seconds * 1000))\n    seconds, millis = divmod(total, 1000)\n    minutes, seconds = divmod(seconds, 60)\n    hours, minutes = divmod(minutes, 60)\n    return f"{hours:02d}:{minutes:02d}:{seconds:02d}{separator}{millis:03d}"\n\n\ndef subtitle_cues(segments):\n    for segment in segments:\n        words = segment.get("words") or []\n        if not words:\n            yield segment\n            continue\n        group = []\n        for word in words:\n            if group and (len(group) >= 12 or word["end"] - group[0]["start"] > 7):\n                yield {**segment, "start": group[0]["start"], "end": group[-1]["end"],\n                       "text": "".join(w["word"] for w in group).strip()}\n                group = []\n            group.append(word)\n        if group:\n            yield {**segment, "start": group[0]["start"], "end": group[-1]["end"],\n                   "text": "".join(w["word"] for w in group).strip()}\n\n\ndef labelled_text(segment):\n    prefix = f"[{segment[\'speaker\']}] " if "speaker" in segment else ""\n    return prefix + segment["text"].strip()\n\n\ndef write_outputs(result, folder):\n    folder = Path(folder)\n    segments = result["segments"]\n    partial = "TRANSCRIPCIÓN PARCIAL — vuelve a ejecutar para completarla.\\n\\n" if not result["complete"] else ""\n    text = "\\n\\n".join(f"[{timestamp(s[\'start\'])}] {labelled_text(s)}" for s in segments)\n    if not segments:\n        text = "No se detectó voz en los bloques procesados."\n    (folder / "transcripcion.txt").write_text(partial + text + "\\n", encoding="utf-8")\n    save_json(folder / "transcripcion.json", {**result, "timestamp_unit": "seconds"})\n    cues = list(subtitle_cues(segments))\n    for extension, separator in (("srt", ","), ("vtt", ".")):\n        blocks = []\n        for index, cue in enumerate(cues, 1):\n            start, end = cue["start"], max(cue["end"], cue["start"] + 0.001)\n            blocks.append(f"{index}\\n{timestamp(start, separator)} --> {timestamp(end, separator)}\\n"\n                          + html.escape(labelled_text(cue)))\n        prefix = "WEBVTT\\n\\n" if extension == "vtt" else ""\n        (folder / f"subtitulos.{extension}").write_text(prefix + "\\n\\n".join(blocks) + "\\n", encoding="utf-8")\n    archive = folder / "transcripcion_whisper.zip"\n    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:\n        for name in ("transcripcion.txt", "transcripcion.json", "subtitulos.srt", "subtitulos.vtt"):\n            bundle.write(folder / name, arcname=name)\n    return archive\n\n\ndef transcribe(audio_path, folder, device="auto", model_factory=None, decoder=None):\n    folder = Path(folder)\n    config = read_json(folder / "config.json")\n    base_path = folder / "transcripcion_base.json"\n    if base_path.exists():\n        previous = read_json(base_path)\n        if previous.get("complete"):\n            print("Transcripción ya guardada: no se repetirá.", flush=True)\n            write_outputs(previous, folder)\n            return previous\n    if decoder is None:\n        from faster_whisper.audio import decode_audio\n        decoder = decode_audio\n    audio = decoder(str(audio_path), sampling_rate=SAMPLE_RATE)\n    duration = len(audio) / SAMPLE_RATE\n    plan = chunk_plan(duration, config["chunk_seconds"])\n    if model_factory is None:\n        from faster_whisper import WhisperModel\n        model_factory = WhisperModel\n    if device == "auto":\n        import ctranslate2\n        device = "cuda" if ctranslate2.get_cuda_device_count() else "cpu"\n    print(f"Motor: {config[\'model\']} · {device} · audio de {duration / 60:.1f} minutos.", flush=True)\n    if device == "cpu":\n        print("Sin GPU: la transcripción puede tardar bastante. T4 gratuita es recomendable si está disponible.", flush=True)\n    model = None\n    detected_language = config["language"]\n    completed_segments = []\n    result = None\n    for chunk in plan:\n        cache_path = folder / f"bloque_{chunk[\'index\']:04d}.json"\n        if cache_path.exists():\n            cached = read_json(cache_path)\n            if cached.get("chunk") != chunk:\n                raise RuntimeError("Un bloque guardado no corresponde al audio. Conserva la carpeta para revisarlo.")\n            print(f"Bloque {chunk[\'index\'] + 1}/{len(plan)} recuperado.", flush=True)\n        else:\n            if model is None:\n                model = model_factory(config["model"], device=device,\n                                      compute_type="int8_float16" if device == "cuda" else "int8")\n            print(f"Transcribiendo bloque {chunk[\'index\'] + 1}/{len(plan)}…", flush=True)\n            excerpt = audio[round(chunk["read_start"] * SAMPLE_RATE):round(chunk["read_end"] * SAMPLE_RATE)]\n            generator, info = model.transcribe(\n                excerpt, language=detected_language, beam_size=5, word_timestamps=True,\n                vad_filter=True, vad_parameters={"min_silence_duration_ms": 500},\n                condition_on_previous_text=False,\n            )\n            raw = []\n            for segment in generator:\n                raw.append({"start": segment.start, "end": segment.end, "text": segment.text,\n                            "words": [{"start": w.start, "end": w.end, "word": w.word,\n                                       "probability": w.probability} for w in segment.words or []]})\n            retained = keep_core_segments(raw, chunk)\n            # No fijar un idioma detectado en un bloque sin voz.\n            language = detected_language or (info.language if retained else None)\n            cached = {"chunk": chunk, "language": language, "segments": retained}\n            save_json(cache_path, cached)\n        detected_language = detected_language or cached.get("language")\n        completed_segments.extend(cached["segments"])\n        result = {"model": config["model"], "language": detected_language,\n                  "duration": duration, "complete": chunk["index"] == len(plan) - 1,\n                  "diarized": False, "segments": completed_segments,\n                  "completed_chunks": chunk["index"] + 1, "total_chunks": len(plan)}\n        save_json(base_path, result)\n        write_outputs(result, folder)\n    print("Transcripción completa y archivos guardados.", flush=True)\n    return result\n\n\ndef assign_speakers(segments, turns):\n    """Asigna por solapamiento temporal usando una diarización global exclusiva."""\n    turns = sorted(turns, key=lambda t: (t["start"], t["end"]))\n    ends = [t["end"] for t in turns]\n\n    def label(start, end):\n        # Exclusive diarization no tiene turnos solapados: ends está ordenado.\n        index = bisect_right(ends, start)\n        best, chosen = 0.0, "SIN_IDENTIFICAR"\n        while index < len(turns) and turns[index]["start"] < max(end, start + 0.001):\n            turn = turns[index]\n            overlap = max(0, min(max(end, start + 0.001), turn["end"]) - max(start, turn["start"]))\n            if overlap > best:\n                best, chosen = overlap, turn["speaker"]\n            index += 1\n        return chosen\n\n    output = []\n    for segment in segments:\n        words = segment.get("words") or []\n        if not words:\n            output.append({**segment, "speaker": label(segment["start"], segment["end"])})\n            continue\n        groups = []\n        for word in words:\n            speaker = label(word["start"], word["end"])\n            if not groups or groups[-1]["speaker"] != speaker:\n                groups.append({"speaker": speaker, "words": []})\n            groups[-1]["words"].append({**word, "speaker": speaker})\n        for group in groups:\n            words = group["words"]\n            output.append({**group, "start": words[0]["start"], "end": words[-1]["end"],\n                           "text": "".join(w["word"] for w in words).strip()})\n    return output\n\n\ndef diarize(audio_path, folder, device="auto"):\n    folder = Path(folder)\n    result = read_json(folder / "transcripcion_base.json")\n    if not result["complete"]:\n        raise RuntimeError("Completa la transcripción antes de separar hablantes.")\n    cached_path = folder / "hablantes.json"\n    if cached_path.exists():\n        cached = read_json(cached_path)\n        if cached.get("model") != DIARIZATION_MODEL:\n            raise RuntimeError("El archivo de hablantes guardado usa otro modelo.")\n        turns = cached["turns"]\n    else:\n        token = os.environ.get("HF_TOKEN", "").strip()\n        if not token:\n            raise RuntimeError("Falta HF_TOKEN para la diarización opcional.")\n        import torch\n        from faster_whisper.audio import decode_audio\n        from pyannote.audio import Pipeline\n        from pyannote.audio.pipelines.utils.hook import ProgressHook\n        if device == "auto":\n            device = "cuda" if torch.cuda.is_available() else "cpu"\n        pipeline = Pipeline.from_pretrained(DIARIZATION_MODEL, token=token)\n        if pipeline is None:\n            raise RuntimeError("No se pudo descargar el modelo. Revisa el token y los permisos de Hugging Face.")\n        pipeline.to(torch.device(device))\n        waveform = torch.from_numpy(decode_audio(str(audio_path), sampling_rate=SAMPLE_RATE)).unsqueeze(0)\n        print("Separando hablantes sobre la grabación completa…", flush=True)\n        with ProgressHook() as hook:\n            output = pipeline({"waveform": waveform, "sample_rate": SAMPLE_RATE}, hook=hook)\n        turns = [{"start": turn.start, "end": turn.end, "speaker": speaker}\n                 for turn, _, speaker in output.exclusive_speaker_diarization.itertracks(yield_label=True)]\n        save_json(cached_path, {"model": DIARIZATION_MODEL, "turns": turns})\n    labelled = {**result, "diarized": True, "diarization_model": DIARIZATION_MODEL,\n                "segments": assign_speakers(result["segments"], turns)}\n    write_outputs(labelled, folder)\n    print("Diarización completa. Los archivos descargables incluyen los hablantes.", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("stage", choices=("transcribe", "diarize"))\n    parser.add_argument("--audio", required=True)\n    parser.add_argument("--folder", required=True)\n    parser.add_argument("--device", default="auto", choices=("auto", "cpu", "cuda"))\n    args = parser.parse_args()\n    try:\n        if args.stage == "transcribe":\n            transcribe(args.audio, args.folder, args.device)\n        else:\n            diarize(args.audio, args.folder, args.device)\n    except Exception as error:\n        token = os.environ.get("HF_TOKEN", "")\n        message = str(error)\n        if token:\n            message = message.replace(token, "[TOKEN OCULTO]")\n        print(f"No se completó {args.stage}: {message[:1500]}", flush=True)\n        raise SystemExit(1) from None\n\n\nif __name__ == "__main__":\n    main()\n'
ASR_LOCK = 'av==16.0.1\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\ncoloredlogs==15.0.1\nctranslate2==4.6.0\nfaster-whisper==1.2.1\nfilelock==4.2.9\nflatbuffers==25.12.19\nfsspec==2026.9.0\nhf-xet==1.8.0\nhuggingface-hub==0.36.0\nhumanfriendly==10.0\nidna==3.20\nmpmath==1.3.0\nnumpy==2.2.6\nonnxruntime==1.23.2\npackaging==26.3\nprotobuf==7.36.2\npyyaml==6.0.3\nrequests==2.34.2\nsetuptools==80.9.0\nsympy==1.14.0\ntokenizers==0.22.1\ntqdm==4.70.1\ntyping-extensions==4.16.0\nurllib3==2.8.0\n'
DIARIZATION_LOCK = 'aiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nalembic==1.20.1\nasteroid-filterbanks==0.4.0\nattrs==26.1.0\nav==16.0.1\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\ncloudpickle==3.1.2\ncoloredlogs==15.0.1\ncolorlog==6.12.0\ncontourpy==1.4.0\nctranslate2==4.6.0\ncycler==0.12.1\neinops==0.8.2\nfaster-whisper==1.2.1\nfilelock==4.2.9\nflatbuffers==25.12.19\nfonttools==4.68.0\nfrozenlist==1.8.0\nfsspec==2026.9.0\ngoogleapis-common-protos==1.76.0\ngrpcio==1.85.1\nhf-xet==1.8.0\nhuggingface-hub==0.36.0\nhumanfriendly==10.0\nidna==3.20\njinja2==3.1.6\njoblib==1.6.0\njulius==0.2.8\nkiwisolver==1.5.1\nlightning==2.6.6\nlightning-utilities==0.15.3\nmako==1.4.3\nmarkdown-it-py==4.2.0\nmarkupsafe==3.0.4\nmatplotlib==3.11.2\nmdurl==0.1.2\nmpmath==1.3.0\nmultidict==7.2.1\nnarwhals==2.28.0\nnetworkx==3.7\nnumpy==2.2.6\nnvidia-cublas-cu12==12.8.4.1\nnvidia-cuda-cupti-cu12==12.8.90\nnvidia-cuda-nvrtc-cu12==12.8.93\nnvidia-cuda-runtime-cu12==12.8.90\nnvidia-cudnn-cu12==9.10.2.21\nnvidia-cufft-cu12==11.3.3.83\nnvidia-cufile-cu12==1.13.1.3\nnvidia-curand-cu12==10.3.9.90\nnvidia-cusolver-cu12==11.7.3.90\nnvidia-cusparse-cu12==12.5.8.93\nnvidia-cusparselt-cu12==0.7.1\nnvidia-nccl-cu12==2.27.3\nnvidia-nvjitlink-cu12==12.8.93\nnvidia-nvtx-cu12==12.8.90\nonnxruntime==1.23.2\nopentelemetry-api==1.45.2\nopentelemetry-exporter-http-transport==0.66b2\nopentelemetry-exporter-otlp==1.45.2\nopentelemetry-exporter-otlp-common==0.66b2\nopentelemetry-exporter-otlp-proto-common==1.45.2\nopentelemetry-exporter-otlp-proto-grpc==1.45.2\nopentelemetry-exporter-otlp-proto-http==1.45.2\nopentelemetry-proto==1.45.2\nopentelemetry-sdk==1.45.2\nopentelemetry-semantic-conventions==0.66b2\noptuna==5.0.0\npackaging==26.3\npandas==3.1.1\npillow==13.0.0\nprimepy==1.3\npropcache==0.5.4\nprotobuf==7.36.2\npyannote-audio==4.0.7\npyannote-core==6.1.0\npyannote-database==6.2.0\npyannote-metrics==4.2.0\npyannote-pipeline==4.1.0\npyannoteai-sdk==0.4.0\npygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\npytorch-lightning==2.6.6\npytorch-metric-learning==2.9.0\npyyaml==6.0.3\nrequests==2.34.2\nrich==15.0.0\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.18.1\nsetuptools==80.9.0\nsortedcontainers==2.4.0\nsqlalchemy==2.1.8\nsympy==1.14.0\nthreadpoolctl==3.8.0\ntokenizers==0.22.1\ntorch==2.8.0\ntorch-audiomentations==0.12.0\ntorch-pitch-shift==1.2.5\ntorchaudio==2.8.0\ntorchcodec==0.7.0\ntorchmetrics==1.9.0\ntqdm==4.70.1\ntriton==3.4.0\ntyping-extensions==4.16.0\ntzdata==2026.6\nurllib3==2.8.0\nyarl==1.25.1\n'


def install_environment(work, lock_text, stage):
    """Entorno separado del kernel de Colab: no hay que reiniciarlo."""
    import subprocess
    import sys
    work = Path(work)
    bootstrap = work / "herramientas"
    bootstrap_env = os.environ.copy()
    bootstrap_env["PYTHONPATH"] = str(bootstrap)
    if not (bootstrap / "uv").is_dir():
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--target",
                        str(bootstrap), "uv==0.13.7"], check=True)
    uv = [sys.executable, "-m", "uv"]
    python = work / "entorno" / "bin" / "python"
    if not python.exists():
        subprocess.run(uv + ["venv", "--python", "3.12", str(work / "entorno")],
                       env=bootstrap_env, check=True)
    requirements = work / f"{stage}.lock"
    requirements.write_text(lock_text, encoding="utf-8")
    marker = work / f"{stage}.instalado"
    fingerprint = hashlib.sha256(lock_text.encode()).hexdigest()
    if not marker.exists() or marker.read_text() != fingerprint:
        print(f"Preparando {stage}; la primera vez puede tardar varios minutos…", flush=True)
        subprocess.run(uv + ["pip", "install", "--python", str(python), "-r", str(requirements)],
                       env=bootstrap_env, check=True)
        marker.write_text(fingerprint)
    return python


def worker_environment(python, token=""):
    import subprocess
    env = os.environ.copy()
    # CTranslate2 debe encontrar CUDA antes de arrancar el proceso.
    library_code = (
        "import site,pathlib; print(':'.join(str(p) for s in site.getsitepackages() "
        "for p in pathlib.Path(s).glob('nvidia/*/lib')))"
    )
    libraries = subprocess.check_output([str(python), "-c", library_code], text=True).strip()
    env["LD_LIBRARY_PATH"] = libraries + ":" + env.get("LD_LIBRARY_PATH", "")
    env["HF_HUB_DISABLE_TELEMETRY"] = "1"
    env["PYANNOTE_METRICS_ENABLED"] = "0"
    if token:
        env["HF_TOKEN"] = token
    return env


def run_free_colab():
    import subprocess
    import shutil
    import types
    from google.colab import drive, files, userdata

    print("Whisper local en Colab: no hay API de pago ni coste por minuto.", flush=True)
    if GUARDAR_EN_DRIVE:
        drive.mount("/content/drive")
        output_root = Path("/content/drive/MyDrive/Transcripciones_Whisper")
    else:
        output_root = Path("/content/Transcripciones_Whisper")
        print("Sin Drive, los avances se pierden cuando Colab elimina la sesión.", flush=True)
    source = RUTA_AUDIO.strip()
    if not source:
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Selecciona un solo archivo de audio o video.")
        source = next(iter(uploaded))
        del uploaded
    audio = Path(source).expanduser().resolve()

    # El código está dentro de este notebook; no se descarga desde una rama mutable.
    work = Path("/content/whisper_gratis_runtime")
    work.mkdir(parents=True, exist_ok=True)
    worker = work / "whisper_gratis.py"
    worker.write_text(WORKER_SOURCE, encoding="utf-8")
    engine = types.ModuleType("whisper_gratis_notebook")
    exec(compile(WORKER_SOURCE, str(worker), "exec"), engine.__dict__)
    folder = engine.prepare_job(audio, output_root, language=None if IDIOMA == "auto" else IDIOMA)
    print(f"Avances y resultados: {folder}", flush=True)

    gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(
        ["nvidia-smi", "-L"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    ).returncode == 0
    lock = ASR_LOCK
    if gpu:
        lock += "\nnvidia-cublas-cu12==12.8.4.1\nnvidia-cudnn-cu12==9.10.2.21\n"
    python = install_environment(work, lock, "transcripcion")
    command = [str(python), "-u", str(worker)]
    common = ["--audio", str(audio), "--folder", str(folder), "--device", "cuda" if gpu else "cpu"]
    # Cada etapa usa un proceso distinto y libera toda su memoria al terminar.
    completed = subprocess.run(command + ["transcribe"] + common, env=worker_environment(python))
    archive = folder / "transcripcion_whisper.zip"
    if completed.returncode:
        print("La transcripción se interrumpió. Reejecuta con el mismo audio para retomar los bloques guardados.")
        if archive.exists():
            print("Puedes descargar lo que se alcanzó a transcribir; está marcado como parcial.")
            files.download(str(archive))
        return

    if SEPARAR_HABLANTES:
        token = os.environ.get("HF_TOKEN", "").strip()
        if not token:
            for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
                try:
                    token = (userdata.get(name) or "").strip()
                except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                    token = ""
                if token:
                    break
        # Hablantes ya guardados no requieren volver a usar la credencial.
        if not token and not (folder / "hablantes.json").exists():
            print("El texto está completo. Falta el token gratuito para separar hablantes.")
            print("Crea un secreto HF_TOKEN y acepta las condiciones del modelo Community-1.")
            print("https://huggingface.co/settings/tokens")
            print("https://huggingface.co/pyannote/speaker-diarization-community-1")
        else:
            try:
                if not (folder / "hablantes.json").exists():
                    python = install_environment(work, DIARIZATION_LOCK, "diarizacion")
                diarized = subprocess.run(command + ["diarize"] + common,
                                          env=worker_environment(python, token))
                if diarized.returncode:
                    print("La diarización falló; se conserva y descarga la transcripción sin hablantes.")
                    print("Revisa HF_TOKEN y los permisos de Community-1. Reejecutar no repite la transcripción.")
            except subprocess.CalledProcessError:
                print("No se pudo preparar la diarización. La transcripción está completa y conservada.")
            finally:
                token = ""
    print(f"Archivos listos en {folder}")
    files.download(str(archive))


try:
    run_free_colab()
except Exception as error:
    print(f"No se completó: {type(error).__name__}: {error}")
    print("Revisa el mensaje anterior. Los bloques ya guardados se recuperan al reejecutar con el mismo audio.")
